# 📊 01 — Exploratory Data Analysis (EDA)

> **Objective**: Understand the Global Superstore dataset before building ML models. Act like a detective — uncover patterns, anomalies, distributions, and relationships.

**Questions to Answer:**
1. How many rows/columns? Missing values? Duplicates?
2. What are the sales, profit, and discount distributions?
3. Which products, categories, and regions perform best?
4. What are the monthly/yearly sales trends?

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# Style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

COLORS = {'primary': '#003B73', 'secondary': '#0074B7', 'success': '#27AE60', 'alert': '#BF212F'}
print("✅ Libraries loaded")

: 

## 2. Load Data

In [ ]:
df = pd.read_csv('../../02_Dataset/cleaned_data/sales_clean.csv', parse_dates=['order_date', 'ship_date'])
print(f"Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 3. Dataset Overview

In [ ]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Rows:        {df.shape[0]:,}")
print(f"Columns:     {df.shape[1]}")
print(f"Date Range:  {df['order_date'].min().date()} → {df['order_date'].max().date()}")
print(f"Countries:   {df['country'].nunique()}")
print(f"Markets:     {df['market'].nunique()}")
print(f"Customers:   {df['customer_id'].nunique()}")
print(f"Products:    {df['product_id'].nunique()}")
print(f"Categories:  {df['category'].nunique()}")
print(f"Total Revenue: ${df['sales'].sum():,.2f}")
print(f"Total Profit:  ${df['profit'].sum():,.2f}")
print("=" * 60)

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

## 4. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct}).query('Missing > 0').sort_values('Missing', ascending=False)

if missing_df.empty:
    print("✅ No missing values found!")
else:
    print(missing_df)
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='YlOrRd', ax=ax)
    ax.set_title('Missing Value Heatmap', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../visualizations/missing_values.png', bbox_inches='tight')
    plt.show()

## 5. Duplicate Check

In [ ]:
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")
if dupes > 0:
    print("⚠️ Removing duplicates...")
    df = df.drop_duplicates()
    print(f"After removal: {df.shape[0]:,} rows")
else:
    print("✅ No duplicates found!")

## 6. Univariate Analysis

Distribution of key numerical variables.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, col, color in zip(axes.flat, ['sales', 'profit', 'quantity', 'discount'],
                           [COLORS['primary'], COLORS['success'], COLORS['secondary'], COLORS['alert']]):
    sns.histplot(df[col], bins=50, kde=True, color=color, ax=ax, edgecolor='white')
    ax.axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():,.2f}')
    ax.axvline(df[col].median(), color='orange', linestyle='--', label=f'Median: {df[col].median():,.2f}')
    ax.set_title(f'Distribution of {col.title()}', fontsize=13, fontweight='bold')
    ax.legend()

plt.suptitle('Univariate Distribution Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../visualizations/univariate_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# Box plots to detect outliers
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, col in zip(axes, ['sales', 'profit', 'quantity', 'discount']):
    sns.boxplot(y=df[col], ax=ax, color=COLORS['secondary'], flierprops={'marker': 'o', 'markersize': 3})
    ax.set_title(f'{col.title()} Outliers', fontweight='bold')
plt.suptitle('Outlier Detection (Box Plots)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../visualizations/outlier_boxplots.png', bbox_inches='tight')
plt.show()

## 7. Category Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Revenue by Category
cat_rev = df.groupby('category')['sales'].sum().sort_values(ascending=True)
cat_rev.plot.barh(ax=axes[0, 0], color=[COLORS['primary'], COLORS['secondary'], COLORS['success']])
axes[0, 0].set_title('Revenue by Category', fontweight='bold')
axes[0, 0].set_xlabel('Revenue ($)')
for i, v in enumerate(cat_rev):
    axes[0, 0].text(v + 20000, i, f'${v:,.0f}', va='center', fontweight='bold')

# Revenue by Sub-Category (Top 10)
sub_rev = df.groupby('sub_category')['sales'].sum().sort_values(ascending=True).tail(10)
sub_rev.plot.barh(ax=axes[0, 1], color=COLORS['secondary'])
axes[0, 1].set_title('Top 10 Sub-Categories by Revenue', fontweight='bold')

# Revenue by Market
mkt_rev = df.groupby('market')['sales'].sum().sort_values(ascending=True)
mkt_rev.plot.barh(ax=axes[1, 0], color=COLORS['primary'])
axes[1, 0].set_title('Revenue by Market', fontweight='bold')

# Revenue by Region (Top 10)
reg_rev = df.groupby('region')['sales'].sum().sort_values(ascending=True).tail(10)
reg_rev.plot.barh(ax=axes[1, 1], color=COLORS['success'])
axes[1, 1].set_title('Top 10 Regions by Revenue', fontweight='bold')

plt.suptitle('Category & Geographic Revenue Analysis', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../visualizations/category_analysis.png', bbox_inches='tight')
plt.show()

## 8. Time Series Analysis

In [ ]:
monthly = df.groupby(df['order_date'].dt.to_period('M')).agg(
    Revenue=('sales', 'sum'), Profit=('profit', 'sum'), Orders=('order_id', 'nunique')
).reset_index()
monthly['order_date'] = monthly['order_date'].dt.to_timestamp()

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

axes[0].fill_between(monthly['order_date'], monthly['Revenue'], alpha=0.3, color=COLORS['primary'])
axes[0].plot(monthly['order_date'], monthly['Revenue'], color=COLORS['primary'], linewidth=2, marker='o', markersize=4)
axes[0].set_title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Revenue ($)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

axes[1].fill_between(monthly['order_date'], monthly['Profit'], alpha=0.3, color=COLORS['success'])
axes[1].plot(monthly['order_date'], monthly['Profit'], color=COLORS['success'], linewidth=2, marker='o', markersize=4)
axes[1].set_title('Monthly Profit Trend', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Profit ($)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

plt.suptitle('Time Series — Revenue & Profit', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../visualizations/time_series_trends.png', bbox_inches='tight')
plt.show()

In [ ]:
# YoY Growth
yearly = df.groupby('order_year').agg(Revenue=('sales','sum'), Profit=('profit','sum'), Orders=('order_id','nunique')).reset_index()
yearly['Revenue_Growth_%'] = yearly['Revenue'].pct_change() * 100
yearly['Profit_Growth_%'] = yearly['Profit'].pct_change() * 100
print("Yearly Performance Summary:")
print(yearly.round(2).to_string(index=False))

## 9. Key EDA Insights

| # | Insight |
|---|---|
| 1 | Revenue shows a **consistent upward trend** year over year |
| 2 | **Technology** leads revenue; **Furniture** has lowest margins |
| 3 | **APAC** and **EU** are the strongest markets |
| 4 | Discounts above 20% correlate with **negative profits** |
| 5 | Sales spike in **Q4** (Sep–Dec) — seasonality detected |
| 6 | Profit distribution is **right-skewed** with many loss-making transactions |

---
✅ **EDA Complete** — Proceed to `02_RFM_Analysis.ipynb`